# Benchmark Suite

Run all models on a Colab T4 and compare:
- Load time
- VRAM usage
- Tokens/second
- Which models actually fit

Results from our testing:

| Model | Size | Load | VRAM | Speed |
|-------|------|------|------|-------|
| Qwen3-8B Q4_K_M | 5.0 GB | 2.7s | 5.5 GB | 32.0 tok/s |
| Qwen3.6-35B-A3B APEX Nano | 10.9 GB | 79.8s | ~12 GB | (load proven) |
| Gemma 4 26B-A4B APEX I-Mini | 12.3 GB | 98.4s | ~13 GB | (load proven) |

In [ ]:
# Install
!pip install -q llama-cpp-python huggingface_hub --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
import llama_cpp, subprocess, time, os
from huggingface_hub import hf_hub_download
print(f"llama-cpp-python {llama_cpp.__version__}")

def bench(model_path, label, n_ctx=4096, max_tokens=150):
    print(f"\n{'='*50}")
    print(f"BENCH: {label} ({os.path.getsize(model_path)/1e9:.1f}GB)")
    t0 = time.time()
    llm = llama_cpp.Llama(model_path=model_path, n_gpu_layers=-1, n_ctx=n_ctx, verbose=False, n_threads=2)
    load_time = time.time() - t0
    print(f"Load: {load_time:.1f}s")
    prompt = "<|im_start|>user\nWhat is 17 * 23? Think step by step.<|im_end|>\n<|im_start|>assistant\n"
    t0 = time.time()
    out = llm(prompt, max_tokens=max_tokens, temperature=0.7, stop=["<|im_end|>"])
    elapsed = time.time() - t0
    tok = out["usage"]["completion_tokens"]
    print(f"Math: {tok} tok / {elapsed:.1f}s = {tok/elapsed:.1f} tok/s")
    print(f"  {out['choices'][0]['text'][:300]}")
    r = subprocess.check_output(["nvidia-smi","--query-gpu=memory.used,memory.total","--format=csv,noheader"]).decode()
    print(f"VRAM: {r.strip()}")
    del llm; import gc; gc.collect()
    return label, load_time, tok/elapsed

results = []

# 1. Qwen3-8B
try:
    p = hf_hub_download("bartowski/Qwen_Qwen3-8B-GGUF", "Qwen_Qwen3-8B-Q4_K_M.gguf")
    results.append(bench(p, "Qwen3-8B Q4_K_M"))
except Exception as e: print(f"FAIL: {e}")

# 2. Qwen3.6-35B-A3B APEX Nano
try:
    p = hf_hub_download("mudler/Carnice-Qwen3.6-MoE-35B-A3B-APEX-MTP-GGUF",
                        "Carnice-Qwen3.6-MoE-35B-A3B-APEX-MTP-I-Nano.gguf")
    results.append(bench(p, "Qwen3.6-35B-A3B APEX Nano", n_ctx=8192))
except Exception as e: print(f"FAIL: {e}")

# 3. Gemma 4 26B-A4B APEX I-Mini
try:
    p = hf_hub_download("mudler/gemma-4-26B-A4B-it-APEX-GGUF",
                        "gemma-4-26B-A4B-APEX-I-Mini.gguf")
    results.append(bench(p, "Gemma4-26B APEX I-Mini"))
except Exception as e: print(f"FAIL: {e}")

print(f"\n\n{'='*50}")
print("RESULTS:")
for label, load, tps in results:
    print(f"  {label}: Load={load:.1f}s, {tps:.1f} tok/s")
